# W13A · 내 쇼핑몰의 추천 모델 출시 심사

딥러닝응용I(추천시스템) 01분반 · 동덕여자대학교 · 2026년 11월 24일

소소상점의 추천 담당자입니다. **어떤 장바구니에서, 어떤 지표를 우선해, 어떤 모델을 출시할까요?**
[수업 슬라이드](https://github.com/lunalab-ai/recommender/releases/download/2026-fall-w13a-r2/w13a-first-recommendation.pdf)
· [과제 1](https://github.com/lunalab-ai/recommender/blob/2026-fall-w13a-r2/assignments/student/w13a-assignment1.md)

기준을 정하고, 과거 주문에서 두 신호를 학습하고, 같은 검증 주문에서 비교합니다.
내 함수를 장바구니에 연결해 사용자에게 보이는 순위까지 확인합니다. CPU에서 진행하세요.


In [ ]:
import base64
import inspect
import io
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import FileLink, display

root = Path("w13a-project-r2").resolve()
root.mkdir(exist_ok=True)
bundle_text = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(bundle_text))) as bundle:
    for asset in ("retail.json", "media.json", "shop-design.template.html", "w13a_design.py"):
        (root / asset).write_bytes(bundle.read(asset))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import w13a_design as design

data = design.load_data(root / "retail.json")
sku = np.array([item["sku"] for item in data["catalog"]])
media = json.loads((root / "media.json").read_text(encoding="utf-8"))
names = {item["sku"]: media.get("labels", {}).get(item["sku"], item["name"]) for item in data["catalog"]}
template = (root / "shop-design.template.html").read_text(encoding="utf-8")
assert (len(data["train"]), len(sku), len(data["validation"])) == (4046, 250, 674)
print("W13A_DESIGN_DATA_READY", len(data["train"]), len(sku), len(data["validation"]))


## 1. 무엇을 우선할까요?

[기준 쇼핑몰](https://github.com/lunalab-ai/recommender/releases/download/2026-fall-w13a-r1/w13a-default-shop.html)에
3단 케이크 스탠드를 담아 봅니다. 가린 상품을 1위와 5위에 맞힌 두 모델을 생각해 보세요.
Hit@5는 5개 안에서 적중했는지, NDCG@5는 맞힌 상품이 얼마나 위에 있는지 반영합니다.
아래에 우선할 지표와 예상 결과를 씁니다. 실험을 보기 전 가설을 기록하세요.
[W05A §5–6](https://github.com/lunalab-ai/recommender/blob/2026-fall-w05a-v2/course/notion/sessions/w05a-model-comparison.md)


In [ ]:
criterion = ""    # "hit_at_5" 또는 "ndcg_at_5"
hypothesis = ""   # 어떤 장바구니에서 어떤 변화가 유리할까요?


## 2. 과거 주문으로 아이템 CF 만들기

행렬 $X$의 한 행은 주문입니다. $X^\top X$의 대각선은 상품마다 주문된 횟수이고,
대각선 밖은 두 상품이 함께 들어간 주문 수입니다. 작은 표를 보고 케이크 스탠드와 찻잔에
해당하는 두 숫자를 찾아보세요. 코사인의 분모가 0인 경우도 정합니다.
[W03B 코사인](https://github.com/lunalab-ai/recommender/blob/2026-fall-w03b/course/notion/sessions/w03b-cf-step-by-step.md)
· [W04B 아이템 CF](https://github.com/lunalab-ai/recommender/blob/2026-fall-w04b/course/notion/sessions/w04b-cf-synthesis.md)


In [ ]:
shared, counts = design.coorders(data)
positions = [list(sku).index(code) for code in ("22423", "22699", "22776")]
display(pd.DataFrame(shared[np.ix_(positions, positions)],
                     index=[names[sku[i]] for i in positions], columns=[names[sku[i]] for i in positions]))
baseline = design.evaluate(data["validation"], design.POPULAR, shared, counts, np.zeros_like(shared), {}, None, sku)
display(pd.DataFrame([baseline]))


In [ ]:
def student_cosine(shared, counts):
    # np.outer로 각 상품 쌍의 주문 수를 곱하고 제곱근을 취합니다.
    denominator = None
    if denominator is None:
        return None
    # 공동 주문 수를 분모로 나눕니다. np.divide의 out과 where를 활용하세요.
    return None


In [ ]:
cosine = student_cosine(shared, counts)
similarity = design.quantized_cosine(cosine, shared.shape) if cosine is not None else None
if similarity is not None:
    anchor = list(sku).index("22776")
    for cutoff in (1, 5):
        likes = np.where(shared[anchor] >= cutoff, similarity[anchor], 0)
        j = int(np.lexsort((sku, -counts, -likes))[0])
        print(f"최소 {cutoff}건 · 이웃 {np.count_nonzero(likes)}개 · 첫 후보 {names[sku[j]]}")
else:
    print("코사인 식을 채운 다음 한 상품의 이웃을 비교하세요.")


## 3. 훈련 주문에서 잠재요인 얻기

이 실습용 주문 데이터에는 고객 ID와 평점이 없습니다. 훈련 주문으로 만든 행렬에서
상품의 잠재 방향을 구합니다. 방향 행렬의 열은 큰 방향부터 정렬되어 있습니다.

상품×차원 행렬에서 k개 열을 골라 상품×상품 내적 행렬을 만드세요. 장바구니의
한 행을 이 행렬과 곱하면 후보 점수가 나옵니다. k=8과 32에서 어떤 상품이 움직일지 예측하세요.
[W06B 절단 SVD](https://github.com/lunalab-ai/recommender/blob/main/course/notion/sessions/w06b-svd-foundations.md)


In [ ]:
directions = design.latent_directions(shared) if similarity is not None else None
if directions is not None:
    print("훈련 주문:", len(data["train"]), " · 전체 방향:", directions.shape)
else:
    print("코사인 실습을 마치면 실제 훈련 주문에서 잠재 방향을 계산합니다.")


In [ ]:
def student_kernel(directions, k):
    # 상위 k개 열을 잘라 상품×차원 배열을 만듭니다.
    factors = None
    if factors is None:
        return None
    # factors와 전치 행렬을 어떤 순서로 곱할까요?
    return None


In [ ]:
kernels = {}
if directions is not None and student_kernel(directions, 8) is not None:
    i = list(sku).index("22423")
    for k in design.RANKS:
        fitted = student_kernel(directions, k)
        kernels[k] = design.quantized_kernel(fitted, shared.shape)
        score = np.maximum(kernels[k][i], 0)
        score[i] = -1
        print(f"잠재 차원 {k} · 점수 행렬 {fitted.shape} · 첫 후보 {names[sku[np.argmax(score)]]}")
else:
    print("상위 방향의 개수를 정하고 장바구니용 점수 행렬을 만드세요.")


## 4. 같은 후보에서 두 점수 결합하기

이미 담은 상품을 먼저 가린 뒤, 각 신호의 남은 양수 최대값을 같은 기준으로 맞춥니다.
제공된 scale_rows는 음수를 0으로 두고 최댓값을 정규화합니다.
weight는 잠재요인 비중(0~100)입니다. 화면에서 각각의 기여를 보여 줄 수 있도록
가중한 두 배열을 돌려줍니다. 결합하면 어느 상품이 더 올라올까요?
[W12B S02·S04](https://github.com/lunalab-ai/recommender/blob/2026-fall-w12b/course/notion/sessions/w12b-hybrid-recommendation.md)


In [ ]:
def student_parts(cf_raw, latent_raw, blocked, weight):
    # np.where로 가린 배열을 각각 design.scale_rows에 넘기세요.
    left = None
    right = None
    if left is None or right is None:
        return None
    # weight=0과 weight=100에서 어느 쪽이 남아야 하나요?
    return None


In [ ]:
probe = np.array([[10, 99, 2]])
ready = similarity is not None and bool(kernels) and (
    student_parts(probe, probe, np.array([[False, True, False]]), 25) is not None
)
if ready:
    checks = design.exercise_vectors(student_cosine, student_kernel, student_parts)
    left, right = checks["mask_and_mix"]
    low, high = checks["endpoints"]
    assert left[0][1] == right[0][1] == 0, "담은 상품을 먼저 가리세요."
    assert low[1] == [[0, 0]] and high[0] == [[0, 0]], "가중치의 양 끝을 확인하세요."
    print("후보 제외와 결합 양 끝 검사 통과")
else:
    print("작성한 세 함수의 결과를 확인하세요.")


## 5. 실험을 직접 설계하고 검증하기

인기순 기준 행에 한 줄씩 추가하고 다음 셀을 실행하세요. 추가하기 전에 예상 결과를 적어 봅니다.

- CF 두 줄은 최소 동시구매 횟수(1, 3, 5)만 다르게 둡니다.
- 잠재요인 두 줄은 차원(8, 16, 32)만 다르게 둡니다.
- 결합 두 줄은 최소 동시구매·차원을 고정하고 잠재요인 비중만 바꿉니다.

CF의 형식 예는 design.Trial("cf", support=3)입니다.
잠재요인 단독은 종류 "svd", rank, latent_weight=100을 씁니다.
결합은 "hybrid", support, rank, 1~99의 latent_weight를 씁니다.
검증에서는 모든 실험에서 같은 주문과 같은 숨긴 상품을 사용합니다.
[W12A 시간순 분할과 가중치](https://github.com/lunalab-ai/recommender/blob/main/course/notion/sessions/w12a-hybrid-recommendation.md)
[W12A 시간순 분할과 가중치](https://github.com/lunalab-ai/recommender/blob/2026-fall-w12a/course/notion/sessions/w12a-hybrid-recommendation.md)


In [ ]:
trials = [
    design.POPULAR,
    # 자신의 실험을 하나씩 여기에 추가하세요.
]


In [ ]:
run_by_id = {trial.id: trial for trial in trials}
if not 1 <= len(trials) <= 10 or len(run_by_id) != len(trials):
    raise ValueError("서로 다른 실험을 최대 10개까지 만들어 주세요.")
ready_metrics = ready and criterion in design.CRITERIA and bool(hypothesis.strip())
args = (shared, counts, similarity, kernels, student_parts, sku)
if ready_metrics:
    metrics = [design.evaluate(data["validation"], trial, *args) for trial in trials]
    display(pd.DataFrame(metrics))
    print("검증 674건 · 한 품목 195건 / 두 품목 이상 479건")
    print("통제 비교:", design.trial_coverage(trials))
else:
    metrics = []
    print("기준·가설을 적고 2~4단계를 완료하면 직접 작성한 모델이 검증표에 나옵니다.")


## 6. 내 함수로 추천 화면 동작시키기

장바구니나 실험을 바꾸면 student_review가 호출됩니다. 방금 검증에 사용한
같은 계산을 연결하고, 5개 상품마다 두 점수의 기여를 비교하세요.


In [ ]:
def student_review(visible, trial):
    # design.ordered_parts에 visible, trial, 검증에서 쓴 *args를 전달하세요.
    return None


In [ ]:
def visible_for(basket):
    index = {str(code): i for i, code in enumerate(sku)}
    return design.order_matrix([[index[code] for code in basket]], len(sku))


def inspect_basket(basket, trial):
    result = student_review(visible_for(basket), trial)
    if result is None:
        raise ValueError("앱과 추천 계산을 먼저 연결하세요.")
    top, left, right = result
    return pd.DataFrame([
        {"순위": n, "상품": names[sku[i]], "상품번호": sku[i],
         "CF 기여": left[0, i] / (100 * design.SCALE),
         "잠재요인 기여": right[0, i] / (100 * design.SCALE)}
        for n, i in enumerate(top[0], 1)
    ])


def show_workspace():
    import ipywidgets as widgets

    choices = [("케이크 스탠드 한 개", ("22423",)),
               ("하트형 스탠드 한 개", ("22776",)),
               ("케이크 스탠드 + 찻잔", ("22423", "22699"))]
    baskets = widgets.Dropdown(options=choices, description="장바구니")
    candidates = widgets.Dropdown(options=list(run_by_id), description="내 실험")
    output = widgets.Output()
    latest = {}

    def change(_):
        output.clear_output(wait=True)
        with output:
            left = inspect_basket(baskets.value, design.CF_REFERENCE).set_index("순위")
            right = inspect_basket(baskets.value, run_by_id[candidates.value]).set_index("순위")
            latest["sku"] = right["상품번호"].tolist()
            display(pd.concat({"현행 CF": left, "내 실험": right}, axis=1))

    for control in (baskets, candidates):
        control.observe(change, names="value")
    display(widgets.VBox([widgets.HBox([baskets, candidates]), output]))
    change(None)
    return baskets, candidates, latest


if ready_metrics and student_review(visible_for(("22423",)), design.POPULAR) is not None:
    workspace = show_workspace()
    print("W13A_DESIGN_APP_READY")
else:
    print("비교 화면을 열려면 실험을 작성하고 화면이 호출할 함수를 연결하세요.")


## 7. 출시 판단 내보내기

선언한 지표의 표를 보고 출시할 실험 ID를 고릅니다. 비교할 다른 실험 하나와 장바구니를
골라 두 모델에서 순위가 달라진 상품번호를 씁니다. 순위 밖은 0으로 기록됩니다.
숫자와 SKU를 근거로 설명하고, 다음에 확인할 한 가지 한계를 적으세요.


In [ ]:
selected_id = ""
comparison_id = ""
evidence_basket = ()   # 예: ("22423",)
observed_sku = ""
decision_note = ""
limitation = ""


In [ ]:
export_ready = False
if ready_metrics and selected_id:
    if not all(design.trial_coverage(trials).values()):
        raise ValueError("인기순·CF·잠재요인·결합의 통제 비교를 완료하세요.")
    if selected_id not in design.preferred(metrics, criterion) or comparison_id not in run_by_id or comparison_id == selected_id:
        raise ValueError("비교할 실험과 미리 고른 지표를 확인하세요.")
    if not 1 <= len(evidence_basket) <= 5 or observed_sku in evidence_basket or observed_sku not in names:
        raise ValueError("장바구니와 관찰할 후보의 상품번호를 확인하세요.")
    if not decision_note.strip() or not limitation.strip():
        raise ValueError("선택 근거와 한계를 적으세요.")
    top_a, top_b = design.compare_basket(evidence_basket, run_by_id[comparison_id], run_by_id[selected_id], *args)

    def rank_or_zero(items):
        return items.index(observed_sku) + 1 if observed_sku in items else 0

    if rank_or_zero(top_a) == rank_or_zero(top_b):
        raise ValueError("두 실험에서 순위가 달라진 상품을 선택하세요.")
    evidence = dict(basket=list(evidence_basket), comparison_run_id=comparison_id, observed_sku=observed_sku,
                    before_rank=rank_or_zero(top_a), after_rank=rank_or_zero(top_b),
                    hypothesis=hypothesis, note=decision_note, limitation=limitation)
    # 두 결과 화면의 SKU 순서가 검증에 쓰인 같은 점수 계산과 일치하는지 검사합니다.
    for trial_id, expected in ((comparison_id, top_a), (selected_id, top_b)):
        linked = inspect_basket(evidence_basket, run_by_id[trial_id])["상품번호"].tolist()
        assert linked == expected, "검증과 앱이 다른 순위 함수를 사용하고 있습니다."
    manifest = design.make_manifest(data, trials, criterion, selected_id, evidence, checks)
    code = "\n\n".join(inspect.getsource(fn).strip() for fn in
                       (student_cosine, student_kernel, student_parts, student_review)) + "\n"
    payload = design.browser_payload(data, trials, metrics, shared, counts, similarity, kernels, manifest, media)
    html = design.render(template, payload)
    (root / "shop.html").write_text(html, encoding="utf-8", newline="\n")
    (root / "w13a-assignment1.zip").write_bytes(design.package(manifest, code, metrics, html))
    export_ready = True
    display(FileLink("w13a-project-r2/shop.html"), FileLink("w13a-project-r2/w13a-assignment1.zip"))
    print("W13A_DESIGN_SUBMISSION_READY", selected_id, evidence["before_rank"], evidence["after_rank"])


파일 링크가 열리지 않으면 Colab 왼쪽 **파일 → w13a-project-r2**에서 ZIP의 메뉴를 열어
다운로드하거나 아래 DOWNLOAD를 True로 바꾸세요. HTML을 열어 장바구니를 다시 바꿔 보세요.
시험 기간의 주문은 학생용 스냅샷에 들어 있지 않습니다.


In [ ]:
DOWNLOAD = False
if DOWNLOAD and export_ready:
    from google.colab import files
    files.download(str(root / "w13a-assignment1.zip"))


## 마무리 질문

1. 담은 상품을 그대로 다시 보여 주면 검증 점수가 어떻게 왜곡될까요?
2. 동시구매 하한을 높이자 어떤 근거가 사라졌나요?
3. 잠재 차원과 결합 비중은 각각 어느 단계에서 쓰였나요?
4. 두 지표가 다른 선택을 지지할 때 어떤 기준으로 출시했나요?
5. 새로운 상품이 들어온다면 어떤 데이터를 다시 확보해야 할까요?

거래 자료 [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online%2Bretail).
시간 누출 [Ji et al.](https://arxiv.org/html/2010.11060v4).
